# Sigma-Sleight — the rigorous teardown 📏🪞
### Is AdaptiveRSI's σ-standardization a signal, or an order-preserving relabel?

The AdaptiveRSI framework defines overbought/oversold as fixed **σ landmarks** in a logit-RSI space, translated back per length via `σ = logit(RSI)·√(n−1)/2`. We test the one falsifiable thing this implies — that length-aware σ-zones read mean-reversion extremes better than fixed 70/30 — and decompose it into four parts, two of which are **exact identities** (no data can change them) and two of which are empirical.

> The reproducible core executes on a synthetic OU tape with a baked-in, single-horizon mean reversion (a real oversold edge) — the ground truth the decomposition recovers. The real SPY/QQQ horse race is in [`../docs/results.md`](../docs/results.md) via `examples/verify.py`. Plain-language companion: [`01_for_the_curious.ipynb`](01_for_the_curious.ipynb). Seven desk beats, same order.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))           # study root (sigma_sleight/ lives there)
sys.path.insert(0, os.path.abspath("../../.."))      # repo root, for quantlab
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from sigma_sleight import data, rsi, signals, decompose

# Offline synthetic tape: a daily close that mean-reverts at a KNOWN rate (one ~17-bar horizon),
# so an RSI oversold-bounce strategy has a real, modest edge and the rescaling null is exact.
# The real verdict (SPY/QQQ horse race) is in ../docs/results.md via examples/verify.py.
close, truth = data.synthetic_prices(seed=15)
print(f"{truth.n_bars} synthetic bars | baked-in mean reversion kappa {truth.kappa} "
      f"(~{truth.horizon:.0f}-bar horizon)")


2520 synthetic bars | baked-in mean reversion kappa 0.06 (~17-bar horizon)


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — do σ-zones beat fixed 70/30? | 🟡 `WEAK` | The motivating fact (70/30 is length-naive) is real, but the σ-transform sold as the fix is a **strictly monotone** map — within a length it renames a constant RSI threshold and moves **zero** trades (max crossing diff 0 on real SPY/QQQ), and its σ-calibrated band beats naive 70/30 in only **1 of 6** real cells. |
| **Tradability** | 🔴 `MIRAGE` | Even a surviving threshold edge is a daily mean-reversion rule paying repeated equity spreads; the framework itself calls its zones "not buy/sell signals," and the σ-band beats a re-optimised constant in **0 of 6** real cells. |
| **Does σ add signal?** | ⚪ `RELABEL` | The σ-standardization and Rescaled-RSI are order-preserving, so every threshold crossing and rank statistic is invariant. |

> **In one sentence:** AdaptiveRSI's length-awareness is a genuinely sound idea wrapped in a σ-transform that, being monotone, only *relabels* — it cannot create the edge the 38-page manifesto implies, and a re-optimised constant matches or beats it.

*(This notebook executes on the offline synthetic tape — where the identities and a baked-in oversold edge are provable. The real SPY/QQQ horse race that earns the stamps above is in [`../docs/results.md`](../docs/results.md).)*

## Beat 1 · The claim, stated precisely

Define the AdaptiveRSI map at length $n$:

$$\sigma(\text{RSI}; n) = \operatorname{logit}\!\left(\tfrac{\text{RSI}}{100}\right)\cdot\frac{\sqrt{n-1}}{2},\qquad \operatorname{logit}(p)=\ln\frac{p}{1-p}.$$

A zone boundary is a fixed σ landmark $s\in\{\pm0.66,\pm1,\pm\sqrt3,\pm2.14\}$, translated to RSI by the inverse $\text{RSI}=100\,\operatorname{expit}(2s/\sqrt{n-1})$. The claim: these length-aware levels read extremes better than the length-blind 70/30.

In [2]:
# The map is a strict monotone bijection at every length -- verified numerically.
for n in (2, 14, 200):
    chk = decompose.monotone_check(n)
    print(f"len {n:>3}: strictly_increasing={chk['strictly_increasing']}  "
          f"min_step={chk['min_step']:.2e}  roundtrip_err={chk['max_roundtrip_err']:.1e}")

len   2: strictly_increasing=True  min_step=4.95e-04  roundtrip_err=1.4e-14
len  14: strictly_increasing=True  min_step=1.78e-03  roundtrip_err=1.4e-14
len 200: strictly_increasing=True  min_step=6.98e-03  roundtrip_err=1.4e-14


## Beat 2 · So what?

Monotonicity is not a footnote — it is the whole result in embryo. A strictly increasing $\sigma(\cdot;n)$ preserves order, so (i) every **threshold crossing** of a σ-zone coincides with a crossing of a fixed RSI level, and (ii) every **rank statistic** (Spearman IC, quantile) is invariant. Any framework feature built on thresholds or ranks therefore inherits *exactly* the information of the plain RSI it relabels. The economic stakes reduce to a single empirical question: does choosing the σ-implied *constant* per length beat 70/30 — and is it better than just re-optimising a constant?

## Beat 3 · How we'd know — the pre-registered protocol

1. **Crossing identity** (`crossing_identity`): max bar-by-bar position difference between the σ-band strategy and its implied constant-RSI strategy. Pre-registered value: **0**.
2. **Zone arithmetic** (`zone_arithmetic`): reproduce the published table from the formula; confirm RSI(14)=70 ⟺ +1.53σ and monotone compression toward 50.
3. **Horse race** (`strategy_compare`, HAC/Reality-Check on the real run): net Sharpe of fixed vs σ-band vs re-optimised constant. `adaptive − reopt ≤ 0` by construction; the open test is `adaptive` vs `fixed`.
4. **Rescale invariance** (`rescale_increment`): rank-IC gap between rescaled RSI(long) and raw RSI(long). Pre-registered value: **0**.

**Mirage line:** Signal beats `WEAK` only if the σ-band beats fixed 70/30 by a margin surviving a White (2000) Reality Check on the re-optimisation, *and* not matched by the re-optimised constant.

## Beat 4 · The teardown

### 4a · Crossing identity — σ-zone ≡ constant RSI band

In [3]:
rows = []
for n, ls in [(2, -1.732), (5, -1.0), (14, -1.0), (50, -2.14)]:
    out = decompose.crossing_identity(close, n, ls)
    rows.append({'length': n, 'lower_sigma': ls, 'implied_RSI': out['implied_lower_rsi'],
                 'max_pos_diff': out['max_position_diff'], 'entries': out['n_entries']})
display(pd.DataFrame(rows).set_index('length').round(3))
print('Every max_pos_diff is exactly 0: within a length, adaptive == a constant.')

,lower_sigma,implied_RSI,max_pos_diff,entries
length,,,,
2,-1.7320,3.0350,0.0000,42
5,-1.0000,26.8940,0.0000,83
14,-1.0000,36.4770,0.0000,33
50,-2.1400,35.1730,0.0000,1


Every max_pos_diff is exactly 0: within a length, adaptive == a constant.


### 4b · Zone arithmetic — the cheat-sheet is a function of n alone

In [4]:
z = decompose.zone_arithmetic([2, 3, 5, 8, 14, 21, 50, 100, 200])
display(z['table'].round(2))
assert z['rsi70_is_1p53_sigma'] and z['compresses_toward_50']
print('RSI(14)=70 <=>', round(z['sigma_at_rsi70_len14'], 4), 'sigma; table is arithmetic, compresses toward 50.')

,consolidation,support_resistance,trend,overbought_oversold
length,,,,
2,78.9200,88.0800,96.9600,98.6300
3,71.7800,80.4400,92.0500,95.3800
5,65.9300,73.1100,84.9700,89.4700
8,62.2200,68.0500,78.7400,83.4500
14,59.0500,63.5200,72.3300,76.6200
21,57.3300,61.0000,68.4500,72.2500
50,54.7000,57.0900,62.1300,64.8300
100,53.3100,55.0100,58.6200,60.5900
200,52.3400,53.5400,56.1100,57.5300


RSI(14)=70 <=> 1.5275 sigma; table is arithmetic, compresses toward 50.


### 4c · The horse race
Long/flat RSI mean reversion (enter below the lower band, exit above 50), 1 bp/turn. The σ-band uses the framework's −√3σ oversold edge; the re-optimised constant is the in-sample best lower level over a grid (a deliberately *generous*, overfit control).

In [5]:
race = {n: decompose.strategy_compare(close, length=n, cost_bps=1.0) for n in (2, 5, 14)}
tbl = pd.DataFrame({
    n: {'fixed_Sharpe': c['fixed']['sharpe'],
        'adaptive_Sharpe': c['adaptive']['sharpe'],
        'reopt_Sharpe': c['reopt']['sharpe'],
        'adaptive_RSI': c['adaptive_implied_lower_rsi'],
        'reopt_lower': c['reopt']['lower'],
        'adaptive_minus_reopt': c['adaptive_vs_reopt_sharpe']}
    for n, c in race.items()}).T
tbl.index.name = 'length'
display(tbl.round(3))
print('adaptive_minus_reopt <= 0 at every length: the sigma-band is one constant the grid contains.')

,fixed_Sharpe,adaptive_Sharpe,reopt_Sharpe,adaptive_RSI,reopt_lower,adaptive_minus_reopt
length,,,,,,
2,0.6270,0.6900,0.9950,3.0350,10.0000,-0.3050
5,0.7520,0.1090,1.0050,15.0330,25.0000,-0.8960
14,0.6770,0.7380,0.9740,27.6720,39.0000,-0.2360


adaptive_minus_reopt <= 0 at every length: the sigma-band is one constant the grid contains.


The `adaptive_minus_reopt` column is non-positive everywhere — exactly as the identity in 4a forces. The σ-calibration cannot beat re-optimising a constant because it *is* a constant. Whether it beats naive fixed 70/30 (the `fixed` vs `adaptive` columns) is the only thing the real tape adds, and it carries the multiple-testing of the grid, which the real run prices with a Reality Check.

### 4d · Rescaled RSI — cross-length monotone invariance
Rescaled RSI(long→target) $=$ `sigma_to_rsi(rsi_to_sigma(RSI(long)))`, a composition of two strict monotone maps, hence a monotone function of RSI(long). Rank IC is therefore invariant.

In [6]:
out = decompose.rescale_increment(close, target_length=14, long_length=70, horizon=5)
for k in ('ic_native_target', 'ic_native_long', 'ic_rescaled_long', 'rescale_ic_gap',
          'incremental_ic_window_over_native', 'n'):
    print(f'  {k:36s} {out[k]:+.4f}' if isinstance(out[k], float) else f'  {k:36s} {out[k]}')
assert abs(out['rescale_ic_gap']) < 1e-9
print('\nrescale_ic_gap is 0 to machine precision: rescaling is a relabel of raw RSI(long).')

  ic_native_target                     -0.2137
  ic_native_long                       -0.2540
  ic_rescaled_long                     -0.2540
  rescale_ic_gap                       +0.0000
  incremental_ic_window_over_native    -0.1459
  n                                    2445

rescale_ic_gap is 0 to machine precision: rescaling is a relabel of raw RSI(long).


## Beat 5 · The verdict

- **Two exact identities.** The crossing diff (4a) and the rescale IC gap (4d) are **0** to machine precision, at every length — the σ-standardization is order-preserving and adds no threshold or rank information.
- **One arithmetic table.** The cheat-sheet (4b) is a deterministic function of $n$; RSI(14)=70 ⟺ +1.527σ with no market input.
- **One empirical leg.** The horse race (4c): the σ-band loses to a re-optimised constant on the synthetic (Δ Sharpe -0.30); on the real tape the only live question is whether it clears fixed 70/30 after a Reality Check.

**Signal `WEAK`, "σ adds signal?" `RELABEL`.** The framework's substance — length-matched thresholds — is real but is a re-statement of RSI arithmetic, not an edge the σ-transform creates.

## Beat 6 · Could you trade it?

The strategy under the framework is daily RSI mean reversion: high turnover, each round-trip charged a real equity spread, to express a few points of mean-reverting tilt. The σ-relabel changes none of the economics — it changes the number on the axis. And the framework's own disclaimer ("not buy/sell signals") concedes the point. Any edge that survives belongs to *RSI mean reversion at a sensible level*, not to AdaptiveRSI; charged costs and corrected for the level-search, it is a strong **`MIRAGE`** candidate. The real-tape numbers and the Reality Check live in [`../docs/results.md`](../docs/results.md).

## Beat 7 · Going further

- **A truly adaptive σ.** Let the σ bands scale with realised vol (the regime idea the framework gestures at but doesn't implement). That breaks monotone invariance across time and *could* add signal — the version worth a real test.
- **Reality-Checked cross-section.** Sweep lengths × tickers; does the σ-implied band beat 70/30 anywhere after White (2000)? `quantlab.stats` has the machinery.
- **Logit-RSI tails.** The framework's other claim is that the logit scale separates extreme readings the bounded scale compresses. For *rank/threshold* use that's invariant (shown here); for *distance* use (e.g. z-scored displacement) it's a real modelling choice — worth its own teardown.

Fork the relabel claim, or find where σ-zones earn their keep. PRs welcome.